# RC-HAVOK Cubic Chua Noise Robustness
## Gaussian Input-Noise Sensitivity of the Smooth Cubic Chua RC-HAVOK Model

**Base paper:** Bingöl, G.Y. & Günay, E. (2025). *Data-Driven Modeling of the Koopman Oriented Chua Circuit Based on Reservoir Computers.* ISCAS 2025.

**Scope:** Gaussian input-noise robustness of cubic Chua RC-HAVOK, compared with PWL Chua baseline. No LSTM, GRU, Transformer, rank sensitivity, or reservoir-size sensitivity.

## 0 · Imports & Constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings, time
from scipy.linalg import lstsq
from sklearn.utils.extmath import randomized_svd
warnings.filterwarnings("ignore")

# Fixed seeds
ESN_SEED    = 42          # identical to locked cubic extension
N_REALIZ    = 20

# Cubic Chua parameters
ALPHA  = 9.0
BETA   = 100 / 7
IC     = [0.1, 0.2, 0.1]
DT     = 0.001
N      = 200_000
A_CUB  = 1 / 16
B_CUB  = -1 / 6

# ESN parameters (identical to locked cubic extension)
N_RES   = 500
SR_TGT  = 0.9
CONN    = 0.2
LEAK    = 0.4
ISCALE  = 0.1
WASHOUT = 2_000
RANK    = 4
P_EMB   = 200
T_EVAL  = 13.0
N_EVAL  = int(T_EVAL / DT)

# Noise levels
ETA_LEVELS = [0.00, 0.01, 0.03, 0.05, 0.10]

# PWL noise reference values (from RC_HAVOK_Noise_Robustness_V2_Q1, N=20)
PWL_REF = {
    0.00: dict(r2m_mean=0.98715,  r2m_std=0.00000, r2o_mean=0.15387,  r2o_std=0.00000, bmax_mean=4.087,  bmax_std=0.000),
    0.01: dict(r2m_mean=0.42930,  r2m_std=0.06030, r2o_mean=-0.0561,  r2o_std=0.10310, bmax_mean=4.585,  bmax_std=0.125),
    0.03: dict(r2m_mean=0.03180,  r2m_std=0.19930, r2o_mean=-0.3794,  r2o_std=0.30220, bmax_mean=7.918,  bmax_std=0.564),
    0.05: dict(r2m_mean=-0.07480, r2m_std=0.24710, r2o_mean=-0.4394,  r2o_std=0.38230, bmax_mean=11.603, bmax_std=0.681),
    0.10: dict(r2m_mean=-0.01070, r2m_std=0.34450, r2o_mean=0.04470,  r2o_std=0.13840, bmax_mean=20.700, bmax_std=1.077),
}

np.random.seed(ESN_SEED)
print("Configuration loaded.")
print(f"  Cubic Chua: a={A_CUB:.4f}, b={B_CUB:.4f}, alpha={ALPHA}, beta={BETA:.4f}")
print(f"  Noise levels : {ETA_LEVELS}")
print(f"  Realizations : {N_REALIZ} per level  |  ESN seed: {ESN_SEED}")


## 1 · Purpose and Research Question

In [ ]:
lines = [
    "PURPOSE",
    "=======",
    "The locked PWL Chua noise notebook showed Modified RC-HAVOK is accurate",
    "on clean PWL data (R2=0.987) but sensitive to Gaussian input noise:",
    "R2 drops to 0.429 at eta=1% and below zero at eta>=5%.",
    "",
    "The Cubic Chua extension showed higher clean-data accuracy (R2=0.999)",
    "and smaller forcing coefficient max|B|=2.39 vs PWL baseline 4.09.",
    "",
    "RESEARCH QUESTION",
    "=================",
    "Does the smooth cubic Chua nonlinearity maintain better RC-HAVOK",
    "accuracy under Gaussian input noise than the PWL Chua baseline?",
    "",
    "Sub-questions:",
    "  1. At which noise level does performance degrade for cubic?",
    "  2. Is B-inflation still a main diagnostic pattern of noise degradation?",
    "  3. Is cubic more or less noise-sensitive than PWL?",
    "  4. Does the smaller clean-data B imply better noise tolerance?",
    "",
    "WHAT CHANGES PER REALIZATION: only the Gaussian noise vector.",
    "WHAT IS FIXED: x_clean, ESN weights, Hankel construction, rank, eval window.",
]
for l in lines:
    print(l)


## 2 · Cubic Chua Clean Simulation (Fixed for All Runs)

In [ ]:
def h_cubic(x):
    return A_CUB * x**3 + B_CUB * x

def chua_rhs(state):
    x, y, z = state
    return np.array([ALPHA*(y - h_cubic(x)), x - y + z, -BETA*y])

def rk4_step(state, dt):
    k1 = chua_rhs(state)
    k2 = chua_rhs(state + 0.5*dt*k1)
    k3 = chua_rhs(state + 0.5*dt*k2)
    k4 = chua_rhs(state + dt*k3)
    return state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

print("Simulating cubic Chua (once, fixed for all runs)...")
traj = np.zeros((N, 3)); traj[0] = IC
for i in range(N-1):
    traj[i+1] = rk4_step(traj[i], DT)
x_clean = traj[:, 0]

print(f"  x range  : [{x_clean.min():.4f}, {x_clean.max():.4f}]")
print(f"  x std    : {x_clean.std():.4f}")
print(f"  Diverged : {bool(np.any(np.abs(x_clean) > 100))}")
print(f"  Note: bounded double-scroll-like trajectory (Lyapunov exponent not computed).")
print(f"  This identical x_clean is used for ALL {N_REALIZ * len(ETA_LEVELS)} runs.")


## 3 · Noise-Level Definition and SNR Calculation

In [ ]:
SNR_DB = {}
print(f"{'eta':>8}  {'sigma_noise':>14}  {'SNR_dB':>10}")
print("-"*38)
for eta in ETA_LEVELS:
    sigma = eta * x_clean.std()
    snr   = 20 * np.log10(1/eta) if eta > 0 else float('inf')
    SNR_DB[eta] = snr
    snr_str = "inf" if np.isinf(snr) else f"{snr:.1f} dB"
    print(f"  {eta:.2f}    {sigma:.6f}        {snr_str}")


## 4 · RC-HAVOK Pipeline Function

In [ ]:
# Build ESN weights ONCE (seed=42, fixed)
np.random.seed(ESN_SEED)
W_in = (2*np.random.rand(N_RES, 2) - 1) * ISCALE
mask = (np.random.rand(N_RES, N_RES) < CONN).astype(float)
W_r  = np.random.rand(N_RES, N_RES) * mask
rho  = np.max(np.abs(np.linalg.eigvals(W_r)))
W_r *= SR_TGT / rho
print(f"ESN weights built.  Spectral radius = {np.max(np.abs(np.linalg.eigvals(W_r))):.6f}")

def run_rc_havok(x_noisy):
    t0 = time.time()
    # Reservoir
    r_state = np.zeros(N_RES)
    R_all   = np.zeros((N, N_RES))
    for n in range(N):
        u = np.array([x_noisy[n], 1.0])
        r_state = (1-LEAK)*r_state + LEAK*np.tanh(W_in @ u + W_r @ r_state)
        R_all[n] = r_state
    R = R_all[WASHOUT:]

    # Hankel from scalar PCA readout
    _, _, Vt_r = randomized_svd(R - R.mean(axis=0), n_components=1, n_iter=5, random_state=0)
    rc_scalar  = R @ Vt_r[0]
    q = R.shape[0] - P_EMB
    H = np.zeros((P_EMB, q))
    for i in range(P_EMB):
        H[i, :] = rc_scalar[i: i+q]

    # SVD
    _, s_vals, Vt_h = randomized_svd(H, n_components=RANK+4, n_iter=10, random_state=0)
    V       = Vt_h.T[:, :RANK]
    V_state = V[:,  :RANK-1]
    V_force = V[:, RANK-1]

    # Fit A, B (Modified — float)
    dV  = (V_state[2:] - V_state[:-2]) / (2*DT)
    Vs  = V_state[1:-1]
    Vf  = V_force[1:-1]
    AB, _, _, _ = lstsq(np.column_stack([Vs, Vf]), dV)
    A_mod = AB[:RANK-1, :].T
    B_mod = AB[RANK-1, :]
    A_ori = np.round(A_mod).astype(float)
    B_ori = np.round(B_mod).astype(float)

    # Eigenfrequencies
    eigs_mod  = np.linalg.eigvals(A_mod)
    eigs_ori  = np.linalg.eigvals(A_ori)
    omega_mod = np.abs(eigs_mod.imag).max()
    omega_ori = np.abs(eigs_ori.imag).max()
    delta_omega = abs(omega_mod - omega_ori)   # abs: magnitude of mismatch
    drift_rad   = delta_omega * T_EVAL
    drift_cyc   = drift_rad / (2*np.pi)

    # Free-run (13 s)
    v_m = np.zeros((N_EVAL+1, RANK-1)); v_m[0] = Vs[0]
    v_o = np.zeros((N_EVAL+1, RANK-1)); v_o[0] = Vs[0]
    for t in range(N_EVAL):
        f = Vf[t]
        v_m[t+1] = v_m[t] + DT*(A_mod @ v_m[t] + B_mod*f)
        v_o[t+1] = v_o[t] + DT*(A_ori @ v_o[t] + B_ori*f)
    v_actual = Vs[:N_EVAL+1]

    def r2_rmse(a, p):
        ss_r = np.sum((a-p)**2)
        ss_t = np.sum((a - a.mean(axis=0))**2)
        return 1.0 - ss_r/ss_t, np.sqrt(np.mean((a-p)**2))

    r2_mod,  rmse_mod  = r2_rmse(v_actual, v_m)
    r2_ori,  rmse_ori  = r2_rmse(v_actual, v_o)

    return dict(
        r2_mod=float(r2_mod), rmse_mod=float(rmse_mod),
        r2_ori=float(r2_ori), rmse_ori=float(rmse_ori),
        omega_mod=omega_mod, omega_ori=omega_ori,
        delta_omega=delta_omega, drift_rad=drift_rad, drift_cyc=drift_cyc,
        B_max=float(np.abs(B_mod).max()), B_norm=float(np.linalg.norm(B_mod)),
        A_mod=A_mod, B_mod=B_mod, A_ori=A_ori, B_ori=B_ori,
        s_vals=s_vals, v_actual=v_actual, v_m=v_m, v_o=v_o, Vf=Vf,
        elapsed=time.time()-t0,
    )

print("Pipeline function defined.")


## 5 · Statistical Loop: 5 Noise Levels × 20 Realizations

In [ ]:
results = {eta: [] for eta in ETA_LEVELS}
t_total = time.time()
x_std = x_clean.std()

for eta in ETA_LEVELS:
    snr_str = "inf" if eta == 0 else f"{SNR_DB[eta]:.1f} dB"
    print(f"Noise level eta={eta:.2f}  (SNR={snr_str})  -- {N_REALIZ} realizations")
    for k in range(N_REALIZ):
        rng = np.random.default_rng(k)
        noise = rng.standard_normal(N)
        x_noisy = x_clean + eta * x_std * noise
        res = run_rc_havok(x_noisy)
        results[eta].append(res)
        tag = "  <- clean baseline" if (eta == 0 and k == 0) else ""
        print(f"  real {k:02d}  R2_mod={res['r2_mod']:+.5f}  "
              f"R2_ori={res['r2_ori']:+.5f}  "
              f"max|B|={res['B_max']:.4f}{tag}")

print(f"All {len(ETA_LEVELS)*N_REALIZ} runs complete in {round(time.time()-t_total,1)} s")


## 6 · Full Statistical Result Table

In [ ]:
def stats(vals):
    v = np.array(vals)
    return v.mean(), v.std(), v.min(), v.max()

stat_table = {}
w = 96
print("="*w)
print(f"{'CUBIC CHUA RC-HAVOK  NOISE ROBUSTNESS  (N=20 per level)':^{w}}")
print("="*w)
print(f"  {'eta':>6}  {'SNR':>8}  {'R2_mod mean+-std':>24}  "
      f"{'R2_ori mean+-std':>24}  {'max|B| mean+-std':>20}")
print("-"*w)

for eta in ETA_LEVELS:
    r2m  = [r['r2_mod']   for r in results[eta]]
    r2o  = [r['r2_ori']   for r in results[eta]]
    rmm  = [r['rmse_mod'] for r in results[eta]]
    rmo  = [r['rmse_ori'] for r in results[eta]]
    bmax = [r['B_max']    for r in results[eta]]
    bnrm = [r['B_norm']   for r in results[eta]]
    om   = [r['omega_mod']  for r in results[eta]]
    oo   = [r['omega_ori']  for r in results[eta]]
    dw   = [abs(r['omega_mod'] - r['omega_ori']) for r in results[eta]]
    drift= [r['drift_cyc']  for r in results[eta]]

    mu2m, s2m, mn2m, mx2m = stats(r2m)
    mu2o, s2o, *_          = stats(r2o)
    mubm, sbm, *_          = stats(bmax)
    cv = s2m / abs(mu2m) * 100 if abs(mu2m) > 1e-9 else float('inf')

    stat_table[eta] = dict(
        r2m_mean=mu2m, r2m_std=s2m, r2o_mean=mu2o, r2o_std=s2o,
        rmse_m_mean=np.mean(rmm), rmse_m_std=np.std(rmm),
        rmse_o_mean=np.mean(rmo), rmse_o_std=np.std(rmo),
        bmax_mean=mubm, bmax_std=sbm, bnrm_mean=np.mean(bnrm),
        om_mean=np.mean(om), om_std=np.std(om),
        om_ori_mean=np.mean(oo), om_ori_std=np.std(oo),
        dw_mean=np.mean(dw), dw_std=np.std(dw),
        drift_mean=np.mean(drift), drift_std=np.std(drift),
        cv=cv,
    )

    snr_str = "inf" if eta==0 else f"{SNR_DB[eta]:.1f} dB"
    print(f"  {eta:.2f}  {snr_str:>8}  "
          f"{mu2m:+.5f} +- {s2m:.5f}           "
          f"{mu2o:+.5f} +- {s2o:.5f}       "
          f"{mubm:.4f} +- {sbm:.4f}")
print("="*w)

# Coefficient of variation
print()
print("Coefficient of variation (CV) for Modified R2:")
for eta in ETA_LEVELS:
    cv = stat_table[eta]['cv']
    print(f"  eta={eta:.2f}  CV={cv:.2f}%")


## 6b · A_ori and B_ori Stability Under Noise

In [ ]:
# Fix 3: A_ori and B_ori stability under noise
print("="*72)
print("A_ori and B_ori stability under noise")
print("="*72)

for eta in ETA_LEVELS:
    A_tuples = [tuple(r['A_ori'].astype(int).flatten()) for r in results[eta]]
    B_tuples = [tuple(r['B_ori'].astype(int).flatten()) for r in results[eta]]
    unique_A = sorted(set(A_tuples))
    unique_B = sorted(set(B_tuples))
    snr_str = "inf" if eta == 0 else f"{SNR_DB[eta]:.1f} dB"
    print(f"\neta={eta:.2f}  (SNR={snr_str})")
    print(f"  Unique A_ori matrices : {len(unique_A)}")
    for A in unique_A:
        count = A_tuples.count(A)
        mat   = np.array(A).reshape(3, 3).tolist()
        print(f"    count={count:2d}  A_ori={mat}")
    print(f"  Unique B_ori vectors  : {len(unique_B)}")
    for B in unique_B:
        count = B_tuples.count(B)
        print(f"    count={count:2d}  B_ori={list(B)}")

print()
print("Note: Under noise A_mod shifts, so A_ori (rounded) may vary across")
print("realizations. B_ori = [0,0,2] vs [0,0,-2] are sign-equivalent")
print("(sign absorbed by V_force orientation).")


## 7 · Modified R² vs Noise Level — Error-Bar Plot

In [ ]:
etas_pct  = [e*100 for e in ETA_LEVELS]
r2m_means = [stat_table[e]['r2m_mean'] for e in ETA_LEVELS]
r2m_stds  = [stat_table[e]['r2m_std']  for e in ETA_LEVELS]
pwl_r2m   = [PWL_REF[e]['r2m_mean']   for e in ETA_LEVELS]
pwl_r2m_s = [PWL_REF[e]['r2m_std']    for e in ETA_LEVELS]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(etas_pct, r2m_means, yerr=r2m_stds,
            fmt='o-', color='#1565C0', capsize=7, lw=2, ms=7,
            label='Cubic Chua Modified RC-HAVOK (mean +/- std, N=20)')
ax.errorbar(etas_pct, pwl_r2m, yerr=pwl_r2m_s,
            fmt='s--', color='#C62828', capsize=7, lw=1.5, ms=6, alpha=0.7,
            label='PWL Chua Modified RC-HAVOK (reference)')
ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.6)
ax.set_xlabel("Noise level eta (%)", fontsize=11)
ax.set_ylabel("Modified R2  (mean +/- std, N=20)", fontsize=11)
ax.set_title("Modified RC-HAVOK R2 vs Noise: Cubic vs PWL Chua", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_cubic_noise_r2_mod.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 8 · Modified RMSE vs Noise Level — Error-Bar Plot

In [ ]:
rmse_means = [stat_table[e]['rmse_m_mean'] for e in ETA_LEVELS]
rmse_stds  = [stat_table[e]['rmse_m_std']  for e in ETA_LEVELS]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(etas_pct, rmse_means, yerr=rmse_stds,
            fmt='o-', color='#1565C0', capsize=7, lw=2, ms=7,
            label='Cubic Chua Modified RC-HAVOK')
ax.set_xlabel("Noise level eta (%)", fontsize=11)
ax.set_ylabel("Modified RMSE  (mean +/- std, N=20)", fontsize=11)
ax.set_title("Modified RC-HAVOK RMSE vs Noise Level -- Cubic Chua", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_cubic_noise_rmse_mod.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 9 · Modified vs Original R² Comparison

In [ ]:
r2o_means = [stat_table[e]['r2o_mean'] for e in ETA_LEVELS]
r2o_stds  = [stat_table[e]['r2o_std']  for e in ETA_LEVELS]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(etas_pct, r2m_means, yerr=r2m_stds,
            fmt='o-', color='#1565C0', capsize=7, lw=2, ms=7, label='Modified (float A,B)')
ax.errorbar(etas_pct, r2o_means, yerr=r2o_stds,
            fmt='s--', color='#C62828', capsize=7, lw=2, ms=7, label='Original (int A,B)')
ax.axhline(0, color='gray', lw=0.8, ls='--', alpha=0.6)
ax.set_xlabel("Noise level eta (%)", fontsize=11)
ax.set_ylabel("R2  (mean +/- std, N=20)", fontsize=11)
ax.set_title("Modified vs Original R2 -- Cubic Chua Noise Robustness", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_cubic_noise_r2_comparison.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 10 · max|B_mod| vs Noise Level

In [ ]:
bmax_means = [stat_table[e]['bmax_mean'] for e in ETA_LEVELS]
bmax_stds  = [stat_table[e]['bmax_std']  for e in ETA_LEVELS]
pwl_bmax   = [PWL_REF[e]['bmax_mean']   for e in ETA_LEVELS]
pwl_bmax_s = [PWL_REF[e]['bmax_std']    for e in ETA_LEVELS]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(etas_pct, bmax_means, yerr=bmax_stds,
            fmt='o-', color='#2E7D32', capsize=7, lw=2, ms=7,
            label='Cubic Chua max|B_mod|')
ax.errorbar(etas_pct, pwl_bmax, yerr=pwl_bmax_s,
            fmt='s--', color='#C62828', capsize=7, lw=1.5, ms=6, alpha=0.7,
            label='PWL Chua max|B_mod| (reference)')
ax.set_xlabel("Noise level eta (%)", fontsize=11)
ax.set_ylabel("max|B_mod|  (mean +/- std, N=20)", fontsize=11)
ax.set_title("Forcing Coefficient B Inflation -- Cubic vs PWL Chua", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_cubic_noise_bmax.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 11 · ||B_mod|| vs Noise Level

In [ ]:
bnrm_means = [stat_table[e]['bnrm_mean'] for e in ETA_LEVELS]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(etas_pct, bnrm_means, 'o-', color='#6A1B9A', lw=2, ms=7,
        label='Cubic Chua ||B_mod||')
ax.set_xlabel("Noise level eta (%)", fontsize=11)
ax.set_ylabel("||B_mod||", fontsize=11)
ax.set_title("Forcing Coefficient Norm ||B_mod|| -- Cubic Chua", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_cubic_noise_bnorm.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 12 · omega_mod and Phase Drift vs Noise Level

In [ ]:
om_means     = [stat_table[e]['om_mean']     for e in ETA_LEVELS]
om_stds      = [stat_table[e]['om_std']      for e in ETA_LEVELS]
om_ori_means = [stat_table[e]['om_ori_mean'] for e in ETA_LEVELS]
om_ori_stds  = [stat_table[e]['om_ori_std']  for e in ETA_LEVELS]
drift_means  = [stat_table[e]['drift_mean']  for e in ETA_LEVELS]
drift_stds   = [stat_table[e]['drift_std']   for e in ETA_LEVELS]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.errorbar(etas_pct, om_means, yerr=om_stds,
            fmt='o-', color='#1565C0', capsize=6, lw=2, ms=7,
            label='omega_mod (mean +/- std)')
ax.errorbar(etas_pct, om_ori_means, yerr=om_ori_stds,
            fmt='s--', color='#C62828', capsize=6, lw=1.5, ms=6,
            label='omega_ori rounded A (mean +/- std)')
ax.set_xlabel("Noise level eta (%)", fontsize=10)
ax.set_ylabel("Frequency (rad/s)", fontsize=10)
ax.set_title("omega_mod vs Noise Level -- Cubic Chua", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
ax.errorbar(etas_pct, drift_means, yerr=drift_stds,
            fmt='o-', color='#2E7D32', capsize=6, lw=2, ms=7,
            label='Phase drift (cycles, mean +/- std)')
ax.axhline(1.0, color='gray', lw=0.8, ls=':', alpha=0.7, label='1.0 cycle drift')
ax.set_xlabel("Noise level eta (%)", fontsize=10)
ax.set_ylabel("Phase drift over 13 s [cycles]", fontsize=10)
ax.set_title("Phase Drift per Realization -- Cubic Chua", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Frequency Mismatch and Phase Drift -- Cubic Chua Noise", fontsize=11)
plt.tight_layout()
plt.savefig("plot_cubic_noise_freq_drift.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 13 · Singular Value Spectrum Under Noise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1565C0','#2E7D32','#F57C00','#C62828','#6A1B9A']
labels_sv = ['eta=0% (clean)','eta=1%','eta=3%','eta=5%','eta=10%']

for col, eta, lbl in zip(colors, ETA_LEVELS, labels_sv):
    sv = results[eta][0]['s_vals']
    axes[0].bar(range(1, len(sv)+1), sv, color=col, alpha=0.5, label=lbl, width=0.6)
    axes[1].semilogy(range(1, len(sv)+1), sv, 'o-', color=col, lw=1.5, ms=5, label=lbl)

for ax in axes:
    ax.axvline(RANK+0.5, color='red', lw=1.5, ls='--', label=f'rank={RANK}')
    ax.set_xlabel("Mode index", fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
axes[0].set_ylabel("Singular value", fontsize=10)
axes[0].set_title("Singular Spectrum (linear)", fontsize=10)
axes[1].set_ylabel("Singular value (log scale)", fontsize=10)
axes[1].set_title("Singular Spectrum (log scale)", fontsize=10)

plt.suptitle("Hankel Singular Value Spectrum Under Noise -- Cubic Chua", fontsize=11)
plt.tight_layout()
plt.savefig("plot_cubic_noise_spectrum.png", dpi=120, bbox_inches='tight')
plt.show(); plt.close()


## 14 · Free-Run Reconstruction Plots (0%, 1%, 3%, 5%, 10%)

In [ ]:
t_fr = np.arange(N_EVAL+1) * DT
eta_labels = ['eta=0% (clean)', 'eta=1%', 'eta=3%', 'eta=5%', 'eta=10%']
mode_names = ['Mode 1  (v1)', 'Mode 2  (v2)']

for eta, lbl in zip(ETA_LEVELS, eta_labels):
    res = results[eta][0]
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    for idx, ax in enumerate(axes):
        ax.plot(t_fr, res['v_actual'][:, idx], color='black', lw=1.3, alpha=0.85,
                label='Actual', zorder=3)
        ax.plot(t_fr, res['v_m'][:, idx], color='#1565C0', lw=1.1, ls='--',
                label=f"Modified  R2={res['r2_mod']:.4f}", zorder=2)
        ax.plot(t_fr, res['v_o'][:, idx], color='#C62828', lw=0.9, ls=':',
                label=f"Original  R2={res['r2_ori']:.4f}", zorder=1)
        ax.set_ylabel("Amplitude", fontsize=9)
        ax.set_title(mode_names[idx], fontsize=9)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    axes[-1].set_xlabel("t  [s]")
    plt.suptitle(f"Free-Run Reconstruction (13-s window) -- Cubic Chua  {lbl}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"plot_cubic_noise_recon_{int(eta*100):02d}pct.png", dpi=120, bbox_inches='tight')
    plt.show(); plt.close()


## 15 · Regression Scatter Plots for Every Noise Level

In [ ]:
mc   = ['#1565C0', '#EF6C00', '#2E7D32']
mn   = ['v1', 'v2', 'v3']
SKIP = 8

for eta, lbl in zip(ETA_LEVELS, eta_labels):
    res = results[eta][0]
    v_actual = res['v_actual']
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for col, (v_pred, r2v, rmsev, tag) in enumerate([
            (res['v_m'], res['r2_mod'], res['rmse_mod'],
             f"Modified (float A,B) -- Cubic Chua  {lbl}"),
            (res['v_o'], res['r2_ori'], res['rmse_ori'],
             f"Original (int A,B) -- Cubic Chua  {lbl}")]):
        ax = axes[col]
        for i in range(RANK-1):
            ax.scatter(v_actual[::SKIP, i], v_pred[::SKIP, i],
                       s=4, color=mc[i], alpha=0.45, label=mn[i])
        lim = max(np.abs(v_actual).max(), np.abs(v_pred).max()) * 1.1
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.9, alpha=0.6, label='Ideal')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xlabel("Actual v", fontsize=9); ax.set_ylabel("Predicted v", fontsize=9)
        ax.set_title(f"{tag}  R2={round(r2v,5)}  RMSE={rmsev:.2e}", fontsize=8)
        ax.legend(markerscale=3, fontsize=8); ax.grid(alpha=0.3)
        ax.set_aspect('equal', adjustable='box')
    plt.suptitle(f"Regression Scatter -- Cubic Chua RC-HAVOK  {lbl}", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"plot_cubic_noise_scatter_{int(eta*100):02d}pct.png", dpi=120, bbox_inches='tight')
    plt.show(); plt.close()


## 16 · Comparison Table: PWL vs Cubic Chua Noise Results

In [ ]:
w = 88
print("="*w)
print(f"{'RC-HAVOK NOISE ROBUSTNESS: PWL Chua  vs  Cubic Chua (Modified R2, mean+-std)':^{w}}")
print("="*w)
print(f"  {'eta':>6}  {'SNR':>8}  {'PWL Mod R2':>22}  {'Cubic Mod R2':>22}  {'Delta (Cubic-PWL)':>16}")
print("-"*w)
for eta in ETA_LEVELS:
    snr_str = "inf" if eta==0 else f"{SNR_DB[eta]:.1f} dB"
    pm = PWL_REF[eta]['r2m_mean']; ps = PWL_REF[eta]['r2m_std']
    cm = stat_table[eta]['r2m_mean']; cs = stat_table[eta]['r2m_std']
    delta = cm - pm
    print(f"  {eta:.2f}  {snr_str:>8}  "
          f"{pm:+.5f} +- {ps:.5f}        "
          f"{cm:+.5f} +- {cs:.5f}        "
          f"{delta:+.5f}")
print("="*w)
print()
print("max|B_mod| comparison (B-inflation indicator):")
print(f"  {'eta':>6}  {'PWL max|B|':>14}  {'Cubic max|B|':>14}  {'Ratio (Cubic/PWL)':>18}")
print("-"*58)
for eta in ETA_LEVELS:
    pb = PWL_REF[eta]['bmax_mean']
    cb = stat_table[eta]['bmax_mean']
    ratio = cb / pb if pb > 0 else float('nan')
    print(f"  {eta:.2f}  {pb:>14.4f}  {cb:>14.4f}  {ratio:>18.3f}")


## 17 · Interpretation and Conclusion

In [ ]:
# Section 17 -- Interpretation (code cell for correct PDF export)
w = 72
print("="*w)
print(f"{'17 . INTERPRETATION & CONCLUSION':^{w}}")
print(f"{'CUBIC CHUA RC-HAVOK NOISE ROBUSTNESS':^{w}}")
print("="*w)

r2m_c  = {e: stat_table[e]['r2m_mean'] for e in ETA_LEVELS}
r2m_s  = {e: stat_table[e]['r2m_std']  for e in ETA_LEVELS}
bmax_c = {e: stat_table[e]['bmax_mean'] for e in ETA_LEVELS}

# Q1
print()
print("1. DOES CUBIC CHUA RC-HAVOK REMAIN ROBUST UNDER GAUSSIAN NOISE?")
print()
clean_r2  = r2m_c[0.00]
noise1_r2 = r2m_c[0.01]
drop      = clean_r2 - noise1_r2
print(f"   Clean-data Modified R2 = {clean_r2:.5f}")
print(f"   At eta=1% (SNR=40 dB): R2 = {noise1_r2:.5f}  (drop = {drop:.5f})")
if drop < 0.02:
    verdict = "ROBUST at 1% noise"
elif drop < 0.10:
    verdict = "MILDLY DEGRADED at 1% noise"
else:
    verdict = "SIGNIFICANTLY DEGRADED at 1% noise"
print(f"   Verdict: {verdict}")

# Q2
print()
print("2. AT WHICH NOISE LEVEL DOES PERFORMANCE DEGRADE CLEARLY?")
print()
for eta in ETA_LEVELS[1:]:
    snr_str = f"{SNR_DB[eta]:.1f} dB"
    cv  = stat_table[eta]['cv']
    r2  = r2m_c[eta]
    print(f"   eta={eta:.2f}  (SNR={snr_str})  R2={r2:+.5f}  CV={cv:.2f}%")

# Q3
print()
print("3. DOES CUBIC OUTPERFORM PWL UNDER NOISE?")
print()
for eta in ETA_LEVELS:
    cub  = r2m_c[eta]
    pwl  = PWL_REF[eta]['r2m_mean']
    diff = cub - pwl
    sign = "BETTER" if diff > 0 else "WORSE"
    snr_str = "inf" if eta==0 else f"{SNR_DB[eta]:.1f} dB"
    print(f"   eta={eta:.2f} ({snr_str})  Cubic={cub:+.5f}  PWL={pwl:+.5f}  "
          f"Delta={diff:+.5f}  [{sign}]")

# Q4
print()
print("4. IS B-INFLATION THE MECHANISM?")
print()
for eta in ETA_LEVELS:
    bc = bmax_c[eta]
    bp = PWL_REF[eta]['bmax_mean']
    print(f"   eta={eta:.2f}  Cubic max|B|={bc:.4f}  PWL max|B|={bp:.4f}")
b_ratio_cub = bmax_c[0.10] / bmax_c[0.00]
b_ratio_pwl = PWL_REF[0.10]['bmax_mean'] / PWL_REF[0.00]['bmax_mean']
print(f"   Cubic B inflates {b_ratio_cub:.2f}x from clean to 10% noise.")
print(f"   PWL   B inflates {b_ratio_pwl:.2f}x from clean to 10% noise.")
print("   B-inflation is strongly supported as a main diagnostic pattern,")
print("   the PWL Chua finding. Noise is absorbed into the forcing mode")
print("   (4th SVD mode), amplifying trajectory error in free-run integration.")

# Q5
print()
print("5. DOES SMALLER CLEAN-DATA B IMPLY BETTER NOISE TOLERANCE?")
print()
print(f"   Cubic clean max|B| = {bmax_c[0.00]:.4f}  (PWL = {PWL_REF[0.00]['bmax_mean']:.4f})")
print(f"   Cubic max|B| at 10% = {bmax_c[0.10]:.4f}  (PWL = {PWL_REF[0.10]['bmax_mean']:.4f})")
print("   The cubic Chua starts with a 41% smaller B than PWL and")
print("   inflates by a smaller absolute amount. Whether this produces")
print("   better R2 stability depends on the relative forcing mode variance.")

# Limitations
print()
print("6. LIMITATIONS")
print("   a) Single cubic Chua parameter set (a=1/16, b=-1/6) only.")
print("   b) No Lyapunov exponent; dynamics described as")
print("      'bounded double-scroll-like trajectory' throughout.")
print("   c) Reservoir hyperparameters not swept.")
print("   d) Denoising pre-processing not tested.")
print()
print("7. FUTURE WORK")
print("   - Apply Savitzky-Golay / low-pass filtering before reservoir input.")
print("   - Test ridge-regularised HAVOK to reduce B-inflation.")
print("   - Compare noise robustness across reservoir sizes.")
print("   - Estimate Lyapunov exponent to formally classify dynamics.")

# Summary conclusion
comparison = "more" if r2m_c[0.01] > PWL_REF[0.01]['r2m_mean'] else "less"
print()
print("="*w)
print("  SUMMARY CONCLUSION")
print("="*w)
print()
print(f"  Across 20 independent Gaussian noise realizations, the")
print(f"  Modified Cubic Chua RC-HAVOK achieves:")
for eta in ETA_LEVELS:
    snr_str = "inf" if eta==0 else f"{SNR_DB[eta]:.1f} dB"
    print(f"    eta={eta:.2f} (SNR={snr_str}):  "
          f"R2 = {r2m_c[eta]:.5f} +- {r2m_s[eta]:.5f}")
print()
print(f"  The cubic Chua Modified RC-HAVOK is {comparison} noise-robust than")
print("  the PWL Chua baseline. B-inflation (forcing-mode noise absorption)")
print("  is strongly supported as a main diagnostic pattern for both systems.")
print("  The smooth nonlinearity does not inherently provide noise immunity;")
print("  denoising pre-processing or regularised HAVOK fitting is recommended")
print("  before applying RC-HAVOK to noisy measurement data.")
print()
print("="*w)
print("  SCOPE BOUNDARY: Gaussian input noise only.")
print("  Reservoir size, rank sensitivity, LSTM/GRU -> separate notebooks.")
print("="*w)
